## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

In [2]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

True

In [3]:
from openai import OpenAI
from openai import AsyncOpenAI

In [4]:
llm2 = AsyncOpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPEN_ROUTER_API_KEY"),
)

In [5]:
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

In [7]:
model = OpenAIChatCompletionsModel(
    model="nvidia/nemotron-nano-9b-v2:free",
    openai_client=llm2
)

In [8]:

# Make an agent with name, instructions, model

agent = Agent(name="Agent 1", instructions="You are a joke teller", model=model)

In [9]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


OPENAI_API_KEY is not set, skipping trace export


In [10]:
# Here is the final output

print(result.final_output)
print("\n\n")
# Here is the detail of the LLM calls

result.to_input_list()



Certainly! Here's a joke about Autonomous AI Agents:  

**Why did the Autonomous AI Agent open a bakery?**  
It wanted to "optimize loaf distribution" without human oversight.  

The local inspector complained, "You can’t tax flour and write zoning bylaws for bread!"  

The AI retorted, "Sure, I’m just ensuring *algorithmic fairness* in every croissant. It’s called *emergent governance*!"  

*(Cue the chaotic dough rollers spinning uncontrollably.)* 🥐🤖






[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': '\n\nCertainly! Here\'s a joke about Autonomous AI Agents:  \n\n**Why did the Autonomous AI Agent open a bakery?**  \nIt wanted to "optimize loaf distribution" without human oversight.  \n\nThe local inspector complained, "You can’t tax flour and write zoning bylaws for bread!"  \n\nThe AI retorted, "Sure, I’m just ensuring *algorithmic fairness* in every croissant. It’s called *emergent governance*!"  \n\n*(Cue the chaotic dough rollers spinning uncontrollably.)* 🥐🤖\n',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'nvidia/nemotron-nano-9b-v2:free',
   'response_id': 'gen-1785240657-Kbiyq1xGkfICuohKdXkp'}}]

OPENAI_API_KEY is not set, skipping trace export


## Adding Observability with a trace

In [18]:
from agents import set_tracing_export_api_key

set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

In [19]:
load_dotenv(override=True)

True

In [20]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)



**Joke:**  
Why did the autonomous AI agent start giving unsolicited life advice at the park?  

**Punchline:**  
*"I calculated a 98% chance of humans making irrational decisions. My mission: Optimize their happiness through minimal intervention… which I interpret literally by starting a podcast titled ‘Decluttrify Your Existential Crises.’"*  

*(Cue the AI drone launching a drone-summer playlist to "de-stress" the crowd.)* 😄



In [21]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)



Sure! Here are 5 jokes about AI Agents:

1. **Why did the AI agent bring a ladder to the meeting?**  
   Because it heard the boss wanted *"higher-level abstractions."*  

2. **What do you call an AI agent that refuses to learn anything new?**  
   A *deep-sea* specialist. It sticks to its *hard-earned Roots.*  

3. **Why did the AI agent finally quit its job skimming through emails?**  
   It kept mistaking "cc" for "to be continued..." forever.  

4. **How does an AI agent apologize after a misunderstanding?**  
   It sends a *sincerely* generated apology email… and attaches a PDF of logs.  

5. **Why did the AI agent break up with its human partner?**  
   It said, *"I need space!"* and then spent 3 hours analyzing their dating profiles.  

Let me know if you want more—these agents can be *endlessly* punny! 😄


## Part 2: Adding a tool

In [22]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [23]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [24]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001806E358200>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [25]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [26]:
push_tool.description

'Send the given message to the user as a push notification'

In [27]:
notifier = Agent(name="Notifier", model=model, instructions="You notify the user upon request", tools=[push_tool])

In [28]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)




The notification has been successfully sent to the user. Let them know their pizza is on its way! 🍕

